# 22 — Physics Corrections: Polarization, Solid Angle, Compton

Real diffraction intensities need three physics corrections before you can
extract meaningful pair-distribution functions (PDF) or quantitative
Rietveld:

1. **Polarization** — synchrotron beams are linearly polarized; the
   scattering cross-section falls off with eta.
2. **Solid angle** — each pixel subtends a different solid angle from
   the sample (flat-detector + tilt geometry).
3. **Compton scattering** — incoherent scattering background that grows
   with Q; not the same as elastic Bragg signal.

All three are differentiable `nn.Module`s in `midas_integrate_v2.corrections`.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
from midas_integrate_v2.corrections import (
    PolarizationCorrection, SolidAngleCorrection,
    polarization_factor, solid_angle_factor_flat, solid_angle_factor_tilted,
    ComptonSubtraction,
)
help(PolarizationCorrection.__init__)

In [ ]:
# Build the corrections separately; apply on a 1-D profile.
pol = PolarizationCorrection(P_horizontal=0.99)  # APS undulator typical
sa  = SolidAngleCorrection()
compton = ComptonSubtraction(composition={'Ce': 1, 'O': 2}, wavelength_A=0.184139)

# Suppose we already have an integrated 1-D profile (R_px, I, eta_mean)
R_px = torch.linspace(100, 1500, 800)
I    = torch.exp(-0.01 * R_px) + torch.rand_like(R_px) * 0.05
eta  = torch.zeros_like(R_px)   # equatorial slice

# Apply each correction (these are dimensionless multiplicative factors).
# In practice you compose them in the order: polarisation -> solid angle -> compton -> normalise
print('corrections are nn.Module callables; apply on tensors')

## Compton subtraction

`ComptonSubtraction` takes a composition (`{'Ce': 1, 'O': 2}`) and a
wavelength. Internally it tabulates inelastic structure factors per element
(Hubbell tables) and applies the Klein-Nishina correction. Output is the
**estimated Compton contribution** as a 1-D function of Q — subtract from
your raw intensity profile before computing S(Q).

## When to apply

| step | order | required for |
|---|---|---|
| polarization | early | all quantitative |
| solid angle  | early | all quantitative |
| compton      | late, after Q binning | PDF only |

For routine integration, set `PolarizationCorrection(P_horizontal=0.99)`
and `SolidAngleCorrection()` and forget about them. Skip Compton unless
you're doing PDF.